# 03 · Modelado

Se comparan cinco modelos: **Regresión logística, Árbol de decisión, Random Forest, Gaussian Naive Bayes y SVM**.

- La comparación se hace con **validación cruzada estratificada (5 folds) solo sobre train**.
  El conjunto de test no se usa en este notebook.
- **Métrica objetivo: PR-AUC** (justificación en la sección 4). ROC-AUC se reporta como apoyo, y
  Recall / Precision / F1 en el umbral que maximiza F1.
- La imputación y el encoding están dentro del `Pipeline`, así que se ajustan en cada fold.

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from src.data import load_dataset, split_train_test
from src.evaluation import compute_metrics
from src.features import engineer
from src.models import CV, best_f1_threshold, cv_evaluate, get_models, tune

pd.set_option("display.width", 140)
X, y = load_dataset()
X = engineer(X)
X_train, X_test, y_train, y_test = split_train_test(X, y)
del X_test, y_test  # el test no se usa en este notebook
print("train:", X_train.shape, f"| positivos {y_train.mean():.2%}")
Path("../artifacts").mkdir(parents=True, exist_ok=True)

# ¿Binario o multiclase? Si hubiera >2 clases solo se dejarían los 2 mejores de GNB/SVM.
n_clases = y_train.nunique()
print("clases en el target:", n_clases, "->", "binario: se prueban GaussianNB y SVM" if n_clases == 2
      else "multiclase: se elegirían los 2 mejores")

## 1. Los modelos nuevos: Gaussian Naive Bayes y SVM

**¿Multiclase?** No. El target es `readmitted == "<30"` (sí/no), así que el problema es **binario** y
se incorporan **los dos modelos pedidos**. (Si el target tuviera ≥3 clases se habrían dejado solo
los dos mejores entre los candidatos, comparados por la métrica objetivo; el código imprime
arriba el número de clases como comprobación.)

**Por qué estos dos:**
- **Gaussian Naive Bayes:** línea base probabilística, muy rápida. Supone variables independientes
  y gaussianas dentro de cada clase; aquí casi todo es one-hot (binario), así que el supuesto se
  viola y se espera un rendimiento modesto. Sirve para saber cuánto aporta un modelo más flexible.
  No admite `class_weight`.
- **SVM (kernel RBF):** frontera no lineal, con `class_weight="balanced"` para el desbalance. Se
  escalan las variables (necesario para el kernel). Entrenar escala entre O(n²) y O(n³), por eso
  su búsqueda de hiperparámetros se hace sobre una submuestra estratificada de train. Se usa
  `decision_function` (un puntaje, no una probabilidad calibrada); la sigmoide aplicada no altera
  ROC-AUC ni PR-AUC.

## 2. Búsqueda de hiperparámetros (solo con train)

`RandomizedSearchCV` (CV 3 folds, métrica **PR-AUC**) con distribuciones **log-uniformes**:

| Hiperparámetro | Modelo | Distribución | Por qué |
|---|---|---|---|
| `var_smoothing` | GaussianNB | log-uniforme [1e-12, 1e-1] | Fracción de la varianza máxima que se suma a todas las varianzas para evitar divisiones por ~0. Su efecto es multiplicativo y el rango abarca 11 órdenes de magnitud. |
| `C` | SVM | log-uniforme [1e-2, 1e2] | Penalización por error: pequeño = margen amplio (más regularizado); grande = ajusta más al train. Se mueve por factores, no por sumas. |
| `gamma` | SVM | log-uniforme [1e-4, 1e-1] | Alcance de cada muestra en el RBF: pequeño = frontera suave; grande = frontera irregular (sobreajuste). |

Con una distribución uniforme normal casi todos los candidatos caerían en la última década y
nunca se exploraría la zona de valores pequeños; la log-uniforme da la misma probabilidad a cada
orden de magnitud. `C` y `gamma` se buscan **juntos** porque interactúan (un `gamma` grande
necesita un `C` menor para no sobreajustar).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from src.evaluation import plot_search, plot_search_heatmap

sns.set_theme(style="whitegrid")
FIG = "../reports/figures/"

t = time.time()
res_nb, best_nb = tune("GaussianNB", X_train, y_train, n_iter=30)
print("GaussianNB", {k: float(v) for k, v in best_nb.items()}, f"({time.time() - t:.0f} s, train completo)")
t = time.time()
res_svm, best_svm = tune("SVM", X_train, y_train, n_iter=30, sample=12000)
print("SVM", {k: float(v) for k, v in best_svm.items()}, f"({time.time() - t:.0f} s, submuestra 12000)")

plot_search(res_nb, ["var_smoothing"], "GaussianNB: búsqueda de var_smoothing", FIG + "busqueda_gnb.png")
plot_search(res_svm, ["C", "gamma"], "SVM: búsqueda de C y gamma", FIG + "busqueda_svm.png")
plot_search_heatmap(res_svm, "C", "gamma", "SVM: PR-AUC según (C, gamma)", FIG + "busqueda_svm_C_gamma.png")
plt.show()

### Lectura de la búsqueda
LECTURA_BUSQUEDA

In [ ]:
# Se fijan los mejores hiperparámetros y se comparan los 5 modelos en igualdad de condiciones
models = get_models()
models["GaussianNB"].set_params(**best_nb)
models["SVM"].set_params(**best_svm)
print(models["GaussianNB"]["model"], models["SVM"]["model"])

## 3. Validación cruzada de los cinco modelos
Los folds (`CV`) son los mismos para todos, así que las métricas por fold son comparables.

In [ ]:
results, oof_store, fold_store = [], {}, []


def evaluate(name, model):
    t = time.time()
    folds, oof = cv_evaluate(model, X_train, y_train, CV, n_jobs=5)
    fold_store.append(folds.assign(modelo=name))
    thr = best_f1_threshold(y_train, oof)
    m = compute_metrics(y_train, oof, thr)
    results.append(
        {
            "modelo": name,
            "roc_auc": folds["roc_auc"].mean(),
            "roc_auc_std": folds["roc_auc"].std(),
            "pr_auc": folds["pr_auc"].mean(),
            "pr_auc_std": folds["pr_auc"].std(),
            "recall": m["recall"],
            "precision": m["precision"],
            "f1": m["f1"],
            "umbral": thr,
            "seg": round(time.time() - t),
        }
    )
    oof_store[name] = oof
    print(f"{name:<14} ROC-AUC {folds['roc_auc'].mean():.4f}  PR-AUC {folds['pr_auc'].mean():.4f}"
          f"  Recall {m['recall']:.3f}  F1 {m['f1']:.3f}  (umbral {thr:.2f}, {results[-1]['seg']} s)")


for name, model in models.items():
    evaluate(name, model)

## 4. Tabla comparativa y elección del ganador
### Métrica objetivo: PR-AUC
- El problema está **desbalanceado** (≈ 9 % positivos). Con tantos negativos, ROC-AUC es
  optimista porque los verdaderos negativos mantienen baja la tasa de falsos positivos; **PR-AUC**
  mide solo lo que importa: qué tan bien se *priorizan* los reingresos reales (precisión vs.
  recall). Su línea base es la tasa de positivos (0.09), no 0.5.
- Operativamente el modelo sirve para **priorizar** pacientes de alto riesgo, es decir, es un
  problema de ranking de la clase positiva: justo lo que resume PR-AUC.
- Recall/Precision/F1 dependen de un umbral; se muestran, pero no se usan para elegir porque el
  umbral se ajusta después. ROC-AUC queda como apoyo / desempate.

El ganador es el de mayor **PR-AUC medio en CV**.

In [ ]:
res = pd.DataFrame(results)
res["ganador"] = ""
res.loc[res["pr_auc"].idxmax(), "ganador"] = "◄"
print(res.round(4).to_string(index=False))
res

In [ ]:
import matplotlib.pyplot as plt  # noqa: E402
import seaborn as sns  # noqa: E402

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for a, (col, err, title) in zip(ax, [("roc_auc", "roc_auc_std", "ROC-AUC (CV 5 folds)"),
                                      ("pr_auc", "pr_auc_std", "PR-AUC (CV 5 folds)")]):
    sns.barplot(data=res, y="modelo", x=col, ax=a, color="#4c72b0")
    a.errorbar(res[col], range(len(res)), xerr=res[err], fmt="none", c="k", capsize=3)
    a.set_title(title)
    a.set_xlim(res[col].min() - 0.03 if col == "roc_auc" else 0, None)
    a.set_ylabel("")
fig.tight_layout()
fig.savefig("../reports/figures/comparacion_modelos_cv.png", dpi=130)

### Variación entre folds: boxplots y líneas

In [ ]:
from src.evaluation import plot_cv_boxplots, plot_cv_lines

folds_all = pd.concat(fold_store, ignore_index=True)
plot_cv_boxplots(folds_all, FIG + "boxplots_cv.png")
plot_cv_lines(folds_all, FIG + "lineas_cv.png")
folds_all.groupby("modelo")[["pr_auc", "roc_auc"]].agg(["mean", "std", "min", "max"]).round(4)

LECTURA_FOLDS

## 3. Guardar resultados para el notebook 04

In [ ]:
win = res.loc[res["pr_auc"].idxmax()]
print("Ganador:", win["modelo"], f"| PR-AUC {win['pr_auc']:.4f} | umbral {win['umbral']:.2f}")

folds_all.to_csv("../artifacts/cv_folds.csv", index=False)
res.drop(columns="ganador").to_csv("../artifacts/cv_results.csv", index=False)
winner = {
    "modelo": win["modelo"],
    "umbral": float(win["umbral"]),
}
with open("../artifacts/winner.json", "w", encoding="utf-8") as f:
    json.dump(winner, f, indent=2, ensure_ascii=False)
winner

## Conclusiones
- **Ganador:** Random Forest (ROC-AUC CV ≈ 0.651, PR-AUC ≈ 0.165).
- La diferencia con la regresión logística (0.648) es muy pequeña, menor que la variación entre
  folds (≈ 0.005-0.007), así que en la práctica rinden parecido. El árbol simple es claramente
  peor (0.622).
- Un ROC-AUC de ~0.65 es normal en este dataset: con la información disponible al alta solo se
  explica una parte del reingreso.